# Model comparison

## Objective

The objective of this notebook is to train, evaluate, and compare multiple baseline classification models using the leakage-safe preprocessing pipeline created in the previous stages.

The analysis will establish a reliable performance baseline for customer churn prediction, compare model behavior using classification metrics such as recall, precision, F1-score, balanced accuracy, and confusion matrices, and identify the most promising models for further tuning.

Particular attention will be given to the churn class (`1`), since correctly identifying customers at risk of leaving is more valuable than maximizing overall accuracy alone.

The results of this notebook will be used to select candidate models for hyperparameter optimization, threshold analysis, business-cost evaluation, and eventual deployment.

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
import numpy as np
import math
from IPython.display import display
import joblib
import matplotlib.pyplot as plt

# Validation
from sklearn.utils.validation import check_is_fitted
from sklearn.base import clone

# Models
from sklearn.ensemble import (ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## Load modeling artifacts

The previously generated modeling artifacts are loaded to prepare the candidate-model evaluation stage.

The training and test splits are restored from the processed data directory, and the `Churn` target is separated from the predictor matrices to reconstruct `X_train`, `y_train`, `X_test`, and `y_test`.

The fitted preprocessing artifact is also loaded so that the same feature-transformation configuration defined in the preprocessing notebook can be reused consistently.

In addition, the baseline cross-validation summary is loaded to preserve the results obtained in the previous modeling stage. These baseline results will later be compared with the candidate-model cross-validation results under the same evaluation framework.

The held-out test set is loaded only as part of the project artifacts and will remain untouched during candidate-model selection and cross-validation.

In [3]:
SPLIT_DATA_DIR = Path("../data/processed/splits")

train_data = pd.read_csv(SPLIT_DATA_DIR / "train.csv")
test_data = pd.read_csv(SPLIT_DATA_DIR / "test.csv")

X_train = train_data.drop(columns="Churn")
y_train = train_data["Churn"]

X_test = test_data.drop(columns="Churn")
y_test = test_data["Churn"]

In [4]:
MODELS_DIR = Path("../models")
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"
if not PREPROCESSOR_PATH.exists():
    raise FileNotFoundError(f"preprocessor not found at {PREPROCESSOR_PATH.resolve()}")
preprocessor = joblib.load(PREPROCESSOR_PATH)

In [5]:
CV_SUMMARY_PATH = Path("../data/processed/cv_summary.csv")

# Load the first CSV column as the row index
baseline_cv_summary = pd.read_csv(
    CV_SUMMARY_PATH,
    index_col=0
)

# Transpose the summary so that:
# rows = models
# columns = metrics
baseline_cv_summary = (
    baseline_cv_summary
    .T
    .reset_index(drop=True)
)

# Identify metric columns
metric_columns = [
    column
    for column in baseline_cv_summary.columns
    if column != "model"
]

# Convert metric columns to numeric
baseline_cv_summary[metric_columns] = baseline_cv_summary[
    metric_columns
].apply(pd.to_numeric)

display(baseline_cv_summary.head())

,model,test_accuracy_mean,test_accuracy_std,train_accuracy_mean,train_accuracy_std,test_balanced_accuracy_mean,test_balanced_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,test_roc_auc_mean,...,train_recall_class_##1##_mean,train_recall_class_##1##_std,test_f1_class_##0##_mean,test_f1_class_##0##_std,train_f1_class_##0##_mean,train_f1_class_##0##_std,test_f1_class_##1##_mean,test_f1_class_##1##_std,train_f1_class_##1##_mean,train_f1_class_##1##_std
0,dummy,0.734647,0.000094,0.734647,0.000024,0.500000,0.000000,0.500000,0.000000,0.500000,...,0.000000,0.000000,0.847027,0.000063,0.847028,0.000016,0.000000,0.000000,0.000000,0.000000
1,logistic_regression,0.802451,0.011979,0.806310,0.003327,0.719843,0.019918,0.725352,0.005413,0.846158,...,0.552843,0.010341,0.869516,0.007640,0.871977,0.002079,0.593078,0.029801,0.602314,0.008377
2,decision_tree,0.728079,0.010033,0.998314,0.000226,0.651277,0.009873,0.996983,0.000436,0.651447,...,0.994147,0.000916,0.814872,0.008071,0.998854,0.000154,0.487588,0.014340,0.996814,0.000429
3,random_forest,0.787718,0.006074,0.998269,0.000294,0.690374,0.009848,0.997487,0.000597,0.818540,...,0.995819,0.001295,0.861381,0.003952,0.998823,0.000200,0.546792,0.015938,0.996736,0.000557


In [6]:
# Row consistency
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

# Expected feature structure
assert X_train.shape[1] == X_test.shape[1]
assert X_train.columns.equals(X_test.columns)

# Target should not be present in X
assert "Churn" not in X_train.columns
assert "Churn" not in X_test.columns

# Binary target validation
assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

# No missing target values
assert y_train.notna().all()
assert y_test.notna().all()

print("Split data validated successfully.")

Split data validated successfully.


### Split data validation

The loaded training and test splits are validated to confirm that the feature and target matrices are structurally consistent.

The checks verify that the number of observations matches between features and targets, both splits contain the same predictor schema, the target variable is excluded from the feature matrices, and the target contains only the expected binary classes.

In [7]:
# Confirm that the preprocessing artifact is fitted
check_is_fitted(preprocessor)

# Confirm expected input feature count
assert hasattr(preprocessor, "feature_names_in_")
assert len(preprocessor.feature_names_in_) == X_train.shape[1]

# Confirm training columns match preprocessor input schema
assert set(preprocessor.feature_names_in_) == set(X_train.columns)

print("Preprocessor validated successfully.")

Preprocessor validated successfully.


### Preprocessor validation

The saved preprocessing artifact is validated to confirm that it remains fitted and expects the same feature schema as the current training data.

A small transformation check is also performed to verify that the loaded preprocessor can successfully transform new observations without refitting.

In [8]:
# Summary must not be empty
assert not baseline_cv_summary.empty

# Model column must exist
assert "model" in baseline_cv_summary.columns

# Model names should be unique
assert baseline_cv_summary["model"].is_unique

# No missing model names
assert baseline_cv_summary["model"].notna().all()

# Metric columns should exist
metric_columns = [
    column
    for column in baseline_cv_summary.columns
    if column != "model"
]

assert len(metric_columns) > 0

# Metric values should be numerical
assert all(
    np.issubdtype(
        baseline_cv_summary[column].dtype,
        np.number
    )
    for column in metric_columns
)

# No missing metric values
assert baseline_cv_summary[
    metric_columns
].notna().all().all()

# No infinite metric values
assert np.isfinite(
    baseline_cv_summary[
        metric_columns
    ].to_numpy()
).all()

# Standard deviation columns
std_columns = [
    column
    for column in metric_columns
    if column.endswith("_std")
]

# Standard deviations cannot be negative
assert (
    baseline_cv_summary[std_columns] >= 0
).all().all()

print("Baseline CV summary validated successfully.")

Baseline CV summary validated successfully.


### Baseline cross-validation summary validation

The previously saved baseline cross-validation summary is validated before being reused for model comparison.

The checks confirm that the summary contains valid model identifiers, the expected metric structure, numerical and finite metric values, and non-negative standard deviations.

This ensures that the baseline results can be compared reliably with the candidate-model cross-validation results.

## Candidate model initialization

A set of additional classification models is initialized for baseline comparison.

These models represent different machine learning families, including tree ensembles, boosting methods, margin-based classifiers, distance-based methods, probabilistic models, and neural networks.

At this stage, the models are created using their default configurations. The objective is not to optimize them yet, but to establish a broad and consistent comparison across different modeling approaches.

In [9]:
candidate_models = {
    "extra_trees": ExtraTreesClassifier(),
    "gradient_boosting": GradientBoostingClassifier(),
    "hist_gradient_boosting": HistGradientBoostingClassifier(),
    "support_vector_machine": SVC(),
    "knn": KNeighborsClassifier(),
    "gaussian_naive_bayes": GaussianNB(),
    "mlp": MLPClassifier(),
    "xgboost": XGBClassifier(),
    "lightgbm": LGBMClassifier(verbose=0),
    "catboost": CatBoostClassifier(verbose=0)
}

In [10]:
for model_name, model in candidate_models.items():
    print(f"\n--- {model_name} ---")
    print(model.get_params())


--- extra_trees ---
{'bootstrap': False, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False}

--- gradient_boosting ---
{'ccp_alpha': 0.0, 'criterion': 'deprecated', 'init': None, 'learning_rate': 0.1, 'loss': 'log_loss', 'max_depth': 3, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}

--- hist_gradient_boosting ---
{'categorical_features': 'from_dtype', 'class_weight': None, 'ea

### Candidate model validation

The candidate models were initialized successfully and their configurations were inspected using `get_params()`.

This validation confirms that each estimator is available in the current environment and exposes the expected model parameters required for later training, evaluation, and hyperparameter tuning.

The validated models are now ready to be incorporated into the same preprocessing and cross-validation workflow used for the baseline models.

### Candidate pipeline validation

Before running cross-validation, the candidate pipelines are validated to confirm that:

- every candidate model has a corresponding pipeline;
- each pipeline contains both preprocessing and model stages;
- the preprocessing step can transform the training data successfully;
- the transformed data preserves the expected number of observations;
- no missing or infinite values are produced by preprocessing;
- each candidate model can be fitted and can generate predictions successfully.

Only the training data is used during this validation so that the held-out test set remains untouched.

### Preprocessing and model input validation

Before cross-validation, the preprocessing workflow is validated independently from the candidate classifiers.

The validation first confirms that the preprocessing configuration can transform the original training features into a numerical, finite, and model-ready feature matrix while preserving the number of observations.

The resulting transformed matrix is then provided directly to each candidate classifier to verify that the processed representation is compatible with the model's input requirements.

This stage does not evaluate predictive performance or generalization. Its purpose is to confirm that the preprocessing produces valid data and that every candidate model can consume the resulting feature representation successfully.

In [11]:
candidate_models_pipeline = {
    model_name: Pipeline([('preprocessing', preprocessor), ('model', estimator)])
    for model_name, estimator in candidate_models.items()
}

In [12]:
preprocessing_check = clone(preprocessor)

X_train_processed_check = preprocessing_check.fit_transform(X_train, y_train)

In [13]:
# Same number of observations
assert X_train_processed_check.shape[0] == X_train.shape[0]

# Features were actually generated
assert X_train_processed_check.shape[1] > 0

# Convert temporarily for validation
if hasattr(X_train_processed_check, "toarray"):
    processed_values = X_train_processed_check.toarray()
else:
    processed_values = np.asarray(X_train_processed_check)

# Numerical output
assert np.issubdtype(processed_values.dtype, np.number)

# No missing values
assert not np.isnan(processed_values).any()

# No infinite values
assert np.isfinite(processed_values).all()

print(
    "Preprocessing validation passed:",
    X_train.shape,
    "->",
    X_train_processed_check.shape
)

Preprocessing validation passed: (5634, 19) -> (5634, 45)


### Validation results

The preprocessing workflow produced a valid numerical feature matrix with the expected number of observations and without missing or infinite values.

The transformed representation was subsequently tested with each candidate classifier. Models that completed fitting and prediction successfully were confirmed to be compatible with the current preprocessing output.

The validated pipelines can now proceed to stratified cross-validation, where predictive performance and generalization will be evaluated.

## Cross-validation strategy

The candidate models are evaluated using stratified k-fold cross-validation on the training dataset.

A 5-fold `StratifiedKFold` strategy is used to divide the training data into five subsets while approximately preserving the original distribution of the `Churn` target in every fold.

During each iteration, four folds are used to fit the complete modeling pipeline and the remaining fold is used as validation data. The validation fold changes at every iteration until each observation has been evaluated once as unseen data.

Because preprocessing is included inside each model pipeline, the preprocessing stage is fitted only on the training portion of each fold. The corresponding validation fold is transformed using the preprocessing parameters learned from that fold's training data. This prevents preprocessing leakage during cross-validation.

The held-out test set is not used during this stage and remains reserved for the final evaluation of the selected model.

In [14]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [15]:
existing_classes = sorted(y_train.unique())

base_metrics = {
    "precision": precision_score,
    "recall": recall_score,
    "f1": f1_score,
}

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

for metric_name, metric_function in base_metrics.items():
    for classes in existing_classes:
        scoring[f"{metric_name}_class_##{classes}##"] = make_scorer(metric_function, pos_label=classes)

cv_results = {}

for model_name, pipeline in candidate_models_pipeline.items():
    cv_results[model_name] = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_strategy,
        scoring=scoring,
        return_train_score=True
    )

    print(f"{model_name} evaluated successfully.")

extra_trees evaluated successfully.
gradient_boosting evaluated successfully.
hist_gradient_boosting evaluated successfully.
support_vector_machine evaluated successfully.
knn evaluated successfully.
gaussian_naive_bayes evaluated successfully.


/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/emiliogarcialopez/projectGitHub/telco-churn-project/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: Con

mlp evaluated successfully.
xgboost evaluated successfully.
lightgbm evaluated successfully.
catboost evaluated successfully.


In [16]:
summary = []

for model_name, results in cv_results.items():
    row = {"model": model_name}

    for key, values in results.items():
        if key.startswith("test_") or key.startswith("train_"):
            row[f"{key}_mean"] = values.mean()
            row[f"{key}_std"] = values.std()

    summary.append(row)

candidate_cv_summary = pd.DataFrame(summary).T
display(candidate_cv_summary.head(8))

,0,1,2,3,4,5,6,7,8,9
model,extra_trees,gradient_boosting,hist_gradient_boosting,support_vector_machine,knn,gaussian_naive_bayes,mlp,xgboost,lightgbm,catboost
test_accuracy_mean,0.768906,0.80334,0.795352,0.804935,0.761804,0.696664,0.79056,0.784347,0.79695,0.798014
test_accuracy_std,0.011425,0.013546,0.009343,0.005969,0.004638,0.009061,0.009604,0.012038,0.012779,0.01168
train_accuracy_mean,0.998314,0.831337,0.895412,0.822817,0.837948,0.69653,0.859203,0.952964,0.897675,0.882632
train_accuracy_std,0.000226,0.003354,0.002496,0.00251,0.004727,0.00135,0.004582,0.0044,0.001852,0.002593
test_balanced_accuracy_mean,0.672227,0.715747,0.704969,0.70786,0.686194,0.745053,0.707478,0.698335,0.708407,0.705072
test_balanced_accuracy_std,0.01493,0.018201,0.014233,0.008611,0.007036,0.009173,0.018525,0.014018,0.01612,0.015546
train_balanced_accuracy_mean,0.996983,0.751732,0.848647,0.732795,0.779787,0.744747,0.79516,0.93359,0.852003,0.825581


In [17]:
display(baseline_cv_summary.head(5))
display(candidate_cv_summary.head(5))

,model,test_accuracy_mean,test_accuracy_std,train_accuracy_mean,train_accuracy_std,test_balanced_accuracy_mean,test_balanced_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,test_roc_auc_mean,...,train_recall_class_##1##_mean,train_recall_class_##1##_std,test_f1_class_##0##_mean,test_f1_class_##0##_std,train_f1_class_##0##_mean,train_f1_class_##0##_std,test_f1_class_##1##_mean,test_f1_class_##1##_std,train_f1_class_##1##_mean,train_f1_class_##1##_std
0,dummy,0.734647,0.000094,0.734647,0.000024,0.500000,0.000000,0.500000,0.000000,0.500000,...,0.000000,0.000000,0.847027,0.000063,0.847028,0.000016,0.000000,0.000000,0.000000,0.000000
1,logistic_regression,0.802451,0.011979,0.806310,0.003327,0.719843,0.019918,0.725352,0.005413,0.846158,...,0.552843,0.010341,0.869516,0.007640,0.871977,0.002079,0.593078,0.029801,0.602314,0.008377
2,decision_tree,0.728079,0.010033,0.998314,0.000226,0.651277,0.009873,0.996983,0.000436,0.651447,...,0.994147,0.000916,0.814872,0.008071,0.998854,0.000154,0.487588,0.014340,0.996814,0.000429
3,random_forest,0.787718,0.006074,0.998269,0.000294,0.690374,0.009848,0.997487,0.000597,0.818540,...,0.995819,0.001295,0.861381,0.003952,0.998823,0.000200,0.546792,0.015938,0.996736,0.000557


,0,1,2,3,4,5,6,7,8,9
model,extra_trees,gradient_boosting,hist_gradient_boosting,support_vector_machine,knn,gaussian_naive_bayes,mlp,xgboost,lightgbm,catboost
test_accuracy_mean,0.768906,0.80334,0.795352,0.804935,0.761804,0.696664,0.79056,0.784347,0.79695,0.798014
test_accuracy_std,0.011425,0.013546,0.009343,0.005969,0.004638,0.009061,0.009604,0.012038,0.012779,0.01168
train_accuracy_mean,0.998314,0.831337,0.895412,0.822817,0.837948,0.69653,0.859203,0.952964,0.897675,0.882632
train_accuracy_std,0.000226,0.003354,0.002496,0.00251,0.004727,0.00135,0.004582,0.0044,0.001852,0.002593


In [18]:
# Transpose candidate summary
candidate_cv_summary = (
    candidate_cv_summary
    .T
    .reset_index(drop=True)
)

# Make sure metric columns are numeric
candidate_metric_columns = [
    column
    for column in candidate_cv_summary.columns
    if column != "model"
]

candidate_cv_summary[candidate_metric_columns] = (
    candidate_cv_summary[candidate_metric_columns]
    .apply(pd.to_numeric)
)

display(candidate_cv_summary.head())

,model,test_accuracy_mean,test_accuracy_std,train_accuracy_mean,train_accuracy_std,test_balanced_accuracy_mean,test_balanced_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,test_roc_auc_mean,...,train_recall_class_##1##_mean,train_recall_class_##1##_std,test_f1_class_##0##_mean,test_f1_class_##0##_std,train_f1_class_##0##_mean,train_f1_class_##0##_std,test_f1_class_##1##_mean,test_f1_class_##1##_std,train_f1_class_##1##_mean,train_f1_class_##1##_std
0,extra_trees,0.768906,0.011425,0.998314,0.000226,0.672227,0.014930,0.996983,0.000436,0.786841,...,0.994147,0.000916,0.848113,0.007485,0.998854,0.000154,0.517035,0.024350,0.996814,0.000429
1,gradient_boosting,0.803340,0.013546,0.831337,0.003354,0.715747,0.018201,0.751732,0.005979,0.848115,...,0.582107,0.013288,0.870832,0.008880,0.889213,0.002153,0.587972,0.029636,0.646782,0.008748
2,hist_gradient_boosting,0.795352,0.009343,0.895412,0.002496,0.704969,0.014233,0.848647,0.004548,0.836436,...,0.748997,0.010778,0.865667,0.006029,0.930177,0.001666,0.570334,0.022762,0.791668,0.005558
3,support_vector_machine,0.804935,0.005969,0.822817,0.002510,0.707860,0.008611,0.732795,0.006130,0.797729,...,0.540970,0.014588,0.873250,0.004031,0.884628,0.001396,0.576711,0.013941,0.618266,0.009141
4,knn,0.761804,0.004638,0.837948,0.004727,0.686194,0.007036,0.779787,0.005797,0.781177,...,0.655853,0.008569,0.839383,0.003615,0.891230,0.003223,0.539027,0.010654,0.682331,0.008961


## Combine baseline and candidate model results

The cross-validation results of the candidate models are summarized using the mean and standard deviation of each training and validation metric across the five folds.

The mean represents the model's average performance, while the standard deviation measures its variability across different data partitions. A lower standard deviation indicates more consistent performance, whereas a larger value suggests that the model is more sensitive to the composition of the folds.

Before combining the results, the candidate summary is reorganized so that each row represents one model and each column represents one evaluation statistic. The metric columns are converted to numeric values to ensure that comparisons, sorting, and calculations can be performed correctly.

The candidate and baseline summaries are then validated to confirm that they contain the same metric columns. Their column order is aligned before concatenation, producing a single comparison table containing all evaluated models.

This combined table will be used to compare:

- Average validation performance.
- Performance variability across folds.
- Differences between training and validation scores.
- Possible overfitting or underfitting.
- Performance on the churn class (`1`).
- Overall discrimination through ROC-AUC and PR-AUC.

The principal model-selection metrics are churn recall, churn F1-score, PR-AUC, and balanced accuracy. Accuracy is included for context but will not be used alone because the target distribution is imbalanced.

In [19]:
all_cv_summary = pd.concat(
    [
        baseline_cv_summary,
        candidate_cv_summary
    ],
    ignore_index=True
)

display(all_cv_summary)

,model,test_accuracy_mean,test_accuracy_std,train_accuracy_mean,train_accuracy_std,test_balanced_accuracy_mean,test_balanced_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,test_roc_auc_mean,...,train_recall_class_##1##_mean,train_recall_class_##1##_std,test_f1_class_##0##_mean,test_f1_class_##0##_std,train_f1_class_##0##_mean,train_f1_class_##0##_std,test_f1_class_##1##_mean,test_f1_class_##1##_std,train_f1_class_##1##_mean,train_f1_class_##1##_std
0,dummy,0.734647,0.000094,0.734647,0.000024,0.500000,0.000000,0.500000,0.000000,0.500000,...,0.000000,0.000000,0.847027,0.000063,0.847028,0.000016,0.000000,0.000000,0.000000,0.000000
1,logistic_regression,0.802451,0.011979,0.806310,0.003327,0.719843,0.019918,0.725352,0.005413,0.846158,...,0.552843,0.010341,0.869516,0.007640,0.871977,0.002079,0.593078,0.029801,0.602314,0.008377
2,decision_tree,0.728079,0.010033,0.998314,0.000226,0.651277,0.009873,0.996983,0.000436,0.651447,...,0.994147,0.000916,0.814872,0.008071,0.998854,0.000154,0.487588,0.014340,0.996814,0.000429
3,random_forest,0.787718,0.006074,0.998269,0.000294,0.690374,0.009848,0.997487,0.000597,0.818540,...,0.995819,0.001295,0.861381,0.003952,0.998823,0.000200,0.546792,0.015938,0.996736,0.000557
4,extra_trees,0.768906,0.011425,0.998314,0.000226,0.672227,0.014930,0.996983,0.000436,0.786841,...,0.994147,0.000916,0.848113,0.007485,0.998854,0.000154,0.517035,0.024350,0.996814,0.000429
5,gradient_boosting,0.803340,0.013546,0.831337,0.003354,0.715747,0.018201,0.751732,0.005979,0.848115,...,0.582107,0.013288,0.870832,0.008880,0.889213,0.002153,0.587972,0.029636,0.646782,0.008748
6,hist_gradient_boosting,0.795352,0.009343,0.895412,0.002496,0.704969,0.014233,0.848647,0.004548,0.836436,...,0.748997,0.010778,0.865667,0.006029,0.930177,0.001666,0.570334,0.022762,0.791668,0.005558
7,support_vector_machine,0.804935,0.005969,0.822817,0.002510,0.707860,0.008611,0.732795,0.006130,0.797729,...,0.540970,0.014588,0.873250,0.004031,0.884628,0.001396,0.576711,0.013941,0.618266,0.009141
8,knn,0.761804,0.004638,0.837948,0.004727,0.686194,0.007036,0.779787,0.005797,0.781177,...,0.655853,0.008569,0.839383,0.003615,0.891230,0.003223,0.539027,0.010654,0.682331,0.008961
9,gaussian_naive_bayes,0.696664,0.009061,0.696530,0.001350,0.745053,0.009173,0.744747,0.002255,0.821208,...,0.847492,0.006337,0.756572,0.009303,0.756590,0.001480,0.597406,0.009343,0.597107,0.002125


In [20]:
assert set(baseline_cv_summary.columns) == set(candidate_cv_summary.columns), (
    "Baseline and candidate summaries do not have the same metric columns."
)

candidate_cv_summary = candidate_cv_summary[
    baseline_cv_summary.columns
]

all_cv_summary = pd.concat(
    [
        baseline_cv_summary,
        candidate_cv_summary],
    ignore_index=True
)

display(all_cv_summary.round(3).T)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
model,dummy,logistic_regression,decision_tree,random_forest,extra_trees,gradient_boosting,hist_gradient_boosting,support_vector_machine,knn,gaussian_naive_bayes,mlp,xgboost,lightgbm,catboost
test_accuracy_mean,0.735,0.802,0.728,0.788,0.769,0.803,0.795,0.805,0.762,0.697,0.791,0.784,0.797,0.798
test_accuracy_std,0.0,0.012,0.01,0.006,0.011,0.014,0.009,0.006,0.005,0.009,0.01,0.012,0.013,0.012
train_accuracy_mean,0.735,0.806,0.998,0.998,0.998,0.831,0.895,0.823,0.838,0.697,0.859,0.953,0.898,0.883
train_accuracy_std,0.0,0.003,0.0,0.0,0.0,0.003,0.002,0.003,0.005,0.001,0.005,0.004,0.002,0.003
test_balanced_accuracy_mean,0.5,0.72,0.651,0.69,0.672,0.716,0.705,0.708,0.686,0.745,0.707,0.698,0.708,0.705
test_balanced_accuracy_std,0.0,0.02,0.01,0.01,0.015,0.018,0.014,0.009,0.007,0.009,0.019,0.014,0.016,0.016
train_balanced_accuracy_mean,0.5,0.725,0.997,0.997,0.997,0.752,0.849,0.733,0.78,0.745,0.795,0.934,0.852,0.826
train_balanced_accuracy_std,0.0,0.005,0.0,0.001,0.0,0.006,0.005,0.006,0.006,0.002,0.017,0.007,0.005,0.004
test_roc_auc_mean,0.5,0.846,0.651,0.819,0.787,0.848,0.836,0.798,0.781,0.821,0.822,0.825,0.837,0.842


### Combined-results verification

The baseline and candidate model summaries were combined successfully. Every row represents one evaluated model, and all models share the same cross-validation metrics and summary statistics.

The combined table provides a consistent basis for comparing model performance and stability. Validation means measure expected generalization performance, validation standard deviations measure consistency across folds, and training–validation differences help identify possible overfitting.

The next analysis will rank the models according to the metrics most relevant to churn detection and identify the strongest candidates for hyperparameter tuning.

## Cross-validation performance and variability comparison

The baseline and candidate models are compared using both their **mean cross-validation performance** and their **variability across folds**.

The mean cross-validation score represents the average predictive performance obtained across the validation folds, while the standard deviation measures how much that performance changes between folds. Evaluating both values is important because a model with a strong average score may still be unreliable if its performance varies substantially across different subsets of the training data.

The comparison focuses primarily on metrics relevant to the churn-detection objective, including **recall for the churn class, F1-score, balanced accuracy, and PR-AUC**, while accuracy and ROC-AUC provide additional context.

For each metric:

- A **higher mean score** indicates better average predictive performance.
- A **lower standard deviation** indicates more stable performance across folds.
- A strong candidate should therefore combine **high validation performance with relatively low variability**.

Model selection will not be based on a single metric. The objective is to identify models that provide a favorable balance between churn detection performance, stability across folds, and generalization behavior. The strongest candidates will then be examined further before hyperparameter tuning and final evaluation on the held-out test set.